# 07 · Full-INT8 model export

**Single responsibility:** quantize with representative real images, inspect operators, verify parity, enforce size, and generate a C header

Run after the preceding numbered notebook unless the inputs already exist. Every generated artifact is written outside the notebook so this stage is reproducible.


In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/base.yaml').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from vww_esp32.config import load_config, resolve_paths, seed_everything

config, ROOT = load_config(ROOT / 'configs/base.yaml')
paths = resolve_paths(config, ROOT)
seed_everything(config['project']['seed'])
ROOT


PosixPath('/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32')

In [2]:
import pandas as pd
import tensorflow as tf
from vww_esp32.preprocessing import representative_dataset, split_manifest

manifest = pd.read_csv(paths['processed'] / 'manifest.csv')
splits = split_manifest(manifest)
model = tf.keras.models.load_model(paths['artifacts'] / 'checkpoints' / 'best.keras')
representative = representative_dataset(splits['train'], tuple(config['preprocessing']['image_size']), config['export']['representative_samples'], config['project']['seed'])

In [3]:
from vww_esp32.exporting import convert_full_integer

tflite_path = paths['artifacts'] / 'models' / config['export']['model_filename']
convert_full_integer(model, representative, tflite_path)
tflite_path

INFO:tensorflow:Assets written to: /var/folders/8s/5p551ggs4fb2kjgvk5kzysnr0000gn/T/tmpkxuh5aac/assets


INFO:tensorflow:Assets written to: /var/folders/8s/5p551ggs4fb2kjgvk5kzysnr0000gn/T/tmpkxuh5aac/assets


Saved artifact at '/var/folders/8s/5p551ggs4fb2kjgvk5kzysnr0000gn/T/tmpkxuh5aac'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  4883226240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4883368592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4883370176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4883232752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4883366304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4883374224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4883372816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4883377392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4883374576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4883375808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4883380736: TensorSpec(shape=

/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1789381657.711155 6216081 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1789381657.711167 6216081 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-09-14 17:27:37.711379: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/8s/5p551ggs4fb2kjgvk5kzysnr0000gn/T/tmpkxuh5aac
2026-09-14 17:27:37.713373: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-09-14 17:27:37.713380: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/8s/5p551ggs4fb2kjgvk5kzysnr0000gn/T/tmpkxuh5aac
I0000 00:00:1789381657.730783 6216081 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not e

PosixPath('/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32/artifacts/models/vww_mobilenetv1_96_int8.tflite')

In [4]:
import json
from vww_esp32.exporting import inspect_tflite

export_info = inspect_tflite(tflite_path)
assert export_info['input']['dtype'] == 'int8'
assert export_info['output']['dtype'] == 'int8'
assert export_info['size_bytes'] <= config['export']['max_model_bytes'], export_info
(paths['artifacts'] / 'reports' / 'export_info.json').write_text(json.dumps(export_info, indent=2))
export_info

/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32/.venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


{'size_bytes': 167976,
 'input': {'name': 'serving_default_image:0',
  'shape': [1, 96, 96, 3],
  'dtype': 'int8',
  'scale': 1.0,
  'zero_point': -128},
 'output': {'name': 'StatefulPartitionedCall_1:0',
  'shape': [1, 1],
  'dtype': 'int8',
  'scale': 0.00390625,
  'zero_point': -128},
 'operators': ['ADD',
  'CONV_2D',
  'DEPTHWISE_CONV_2D',
  'FULLY_CONNECTED',
  'LOGISTIC',
  'MEAN',
  'MUL']}

Only builtin integer operators are allowed by the converter. The exact list below must match the firmware resolver; update both together.

In [5]:
export_info['operators']

['ADD',
 'CONV_2D',
 'DEPTHWISE_CONV_2D',
 'FULLY_CONNECTED',
 'LOGISTIC',
 'MEAN',
 'MUL']

In [6]:
import numpy as np
from vww_esp32.exporting import run_tflite

sample = splits['test'].sample(n=min(100, len(splits['test'])), random_state=config['project']['seed'])
def host_preprocess(path):
    decoded = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(decoded, tuple(config['preprocessing']['image_size']), method='bilinear', antialias=False).numpy()
images = np.stack([host_preprocess(path) for path in sample.image_path]).astype(np.float32)
float_prob = model.predict(images, verbose=0).reshape(-1)
int8_prob = run_tflite(tflite_path, images)
parity = {'samples': len(images), 'mean_absolute_probability_error': float(np.mean(np.abs(float_prob-int8_prob))), 'max_absolute_probability_error': float(np.max(np.abs(float_prob-int8_prob))), 'decision_agreement_at_0_5': float(np.mean((float_prob >= .5) == (int8_prob >= .5)))}
assert parity['mean_absolute_probability_error'] < 0.03, parity
parity

2026-09-14 17:27:44.011544: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


{'samples': 100,
 'mean_absolute_probability_error': 0.00897739052772522,
 'max_absolute_probability_error': 0.03563189506530762,
 'decision_agreement_at_0_5': 0.98}

In [7]:
from vww_esp32.exporting import write_c_header

header = ROOT / 'firmware' / 'esp32_cam_vww' / 'include' / config['export']['header_filename']
write_c_header(tflite_path, header, config['export']['c_array_name'])
{'header': str(header), 'bytes': header.stat().st_size}

{'header': '/Volumes/VM_SSD/Machine Learning/visual-wake-word-esp32/firmware/esp32_cam_vww/include/model_data.h',
 'bytes': 1036087}